# ReciclaBot · Proyecto Fase 1
**Materia:** Desarrollo de aplicaciones con Inteligencia Artificial
**Estudiante:** Luis Angel Guerrero
**Avance de proyecto 1 · Semana 2**

ReciclaBot es un asistente conversacional **basado en reglas**, desarrollado con **Python 3.10+ y NLTK**, que orienta a las personas sobre cómo separar correctamente los residuos que se generan en casa (orgánicos, reciclables, no reciclables, de manejo especial y peligrosos) y a dónde llevarlos.

> La información es orientativa. Las reglas de separación, los colores de los contenedores y los horarios de recolección dependen de cada municipio.

## Cómo usar este notebook
1. Ejecuta las celdas en orden con el botón ▶ o con **Entorno de ejecución → Ejecutar todas**.
2. Revisa la validación de ejemplos (sección 3) y las doce conversaciones de prueba (sección 4).
3. Activa la conversación manual (sección 5) para probar tus propias consultas.
4. Descarga las evidencias en la sección 6 y toma capturas de pantalla de las salidas para el PDF.
5. Guarda el notebook en GitHub con **Archivo → Guardar una copia en GitHub**.

## 1. Definición del problema y de la solución

### Problema seleccionado
En los hogares la separación de residuos es confusa: muchas personas no saben en qué contenedor va cada residuo («¿el unicel se recicla?», «¿dónde tiro las pilas?»), las reglas cambian según el municipio y la información oficial está dispersa. Como consecuencia, materiales reciclables se contaminan con residuos orgánicos y terminan en rellenos sanitarios, y residuos peligrosos (pilas, medicamentos, aceites) se mezclan con la basura común.

| Elemento | Descripción |
|---|---|
| **Situación** | Separación incorrecta de los residuos domésticos por falta de orientación inmediata, clara y en lenguaje cotidiano. |
| **Usuarios afectados** | Personas y familias que generan residuos en casa; de forma indirecta, el personal de recolección y los centros de acopio que reciben material contaminado. |
| **Necesidad por atender** | Contar con una orientación inmediata y consistente sobre cómo separar cada tipo de residuo y a dónde llevarlo. |

### Elementos de la solución
- **Objetivo general:** diseñar e implementar un asistente conversacional basado en reglas que oriente a las personas sobre la separación correcta de los residuos domésticos, respondiendo consultas en lenguaje natural de forma inmediata y consistente.
- **Usuarios objetivo:** personas responsables de la basura del hogar, con acceso a un dispositivo con internet, que desean separar correctamente pero no saben cómo. Usuarios secundarios: escuelas y comités vecinales que promueven la separación.
- **Alcance (Fase 1):** chatbot de texto basado en reglas con 18 intenciones sobre separación de residuos, centros de acopio, recolección y consejos. Brinda orientación general (no específica por municipio); no realiza trámites, no consulta calendarios reales, no identifica residuos por imagen y no conserva el contexto entre turnos. Las fases siguientes incorporarán clasificación de intenciones con PLN, clasificación de imágenes de residuos e integración web.
- **Beneficios esperados:** orientación disponible en todo momento y sin esperas; respuestas consistentes alineadas con las categorías de separación (orgánicos, inorgánicos reciclables, inorgánicos no reciclables y manejo especial); menos reciclables contaminados y menos residuos peligrosos en la basura común; una base reutilizable para integrar el reconocimiento de imágenes. Estos beneficios deberán verificarse con usuarios reales en etapas posteriores.

### Justificación del uso de IA
Las consultas sobre residuos se expresan de muchas maneras distintas («¿dónde tiro las pilas?», «pilas usadas», «tengo baterías viejas»). Un asistente con técnicas de procesamiento de lenguaje natural (normalización, tokenización, eliminación de palabras vacías y *stemming* con NLTK) permite asociar expresiones diversas a una misma intención y atender a muchas personas al mismo tiempo sin personal dedicado. En las fases posteriores, la clasificación automática de texto e imágenes ampliará la cobertura del asistente.

## 2. Arquitectura conceptual

```
 ┌──────────┐  texto de la consulta   ┌───────────────────────────────────────────┐
 │          │ ──────────────────────▶ │  CHATBOT (motor de reglas)                │
 │ USUARIO  │                         │  1. Normalización (minúsculas, sin acentos)│
 │          │ ◀────────────────────── │  2. Tokenización + stopwords + stemming   │
 └──────────┘   respuesta en texto    │     (NLTK)                                │
                                      │  3. Identificación de la intención        │
                                      │  4. Selección de la respuesta             │
                                      └───────────────┬───────────────────────────┘
                                             consulta │ ▲ intención + respuesta
                                                      ▼ │
                                      ┌───────────────────────────────────────────┐
                                      │  BASE DE CONOCIMIENTO                     │
                                      │  intención · claves/patrón · respuesta    │
                                      └───────────────────────────────────────────┘
```

| Componente | Función | Información que intercambia |
|---|---|---|
| **Usuario** | Escribe consultas en lenguaje natural sobre residuos. | Envía texto libre; recibe la respuesta del chatbot. |
| **Chatbot (motor de reglas)** | Normaliza y procesa el texto con NLTK, identifica la intención y selecciona la respuesta. | Recibe el texto; consulta la base de conocimiento; devuelve la intención y la respuesta. |
| **Base de conocimiento** | Lista de intenciones con descripción, ejemplos, palabras clave o patrón y respuesta. | Entrega claves y patrones para la identificación y la respuesta asociada a la intención ganadora. |
| **Flujo conversacional** | Define la secuencia de la interacción: saludo → consulta → identificación → respuesta → ¿otra consulta? → despedida. | Controla el inicio, la continuidad y el cierre del diálogo, y el registro de evidencias (entrada, intención, respuesta). |

**Flujo conversacional:** la secuencia de cada turno es *entrada del usuario → identificación de la intención → respuesta del chatbot*. Primero se evalúan las intenciones temáticas (puntuación por claves); si ninguna coincide, se evalúan las reglas conversacionales (expresiones regulares); si tampoco coinciden, se responde con la intención `desconocida`. El diálogo termina cuando se identifica la intención `despedida`.

In [2]:
import sys
assert sys.version_info >= (3, 10), 'Se requiere Python 3.10 o superior.'
%pip -q install nltk==3.9.1
import nltk
nltk.download('stopwords', quiet=True)   # lista de palabras vacías en español
print('Python:', sys.version.split()[0], '| NLTK:', nltk.__version__)

Note: you may need to restart the kernel to use updated packages.
Python: 3.12.1 | NLTK: 3.9.1


## 3. Base de conocimiento y diseño conversacional
La base de conocimiento es una lista de **intenciones temáticas**. Cada intención tiene un identificador, una descripción, ejemplos de consulta, palabras y frases clave y la respuesta asociada. Las palabras clave se convierten en raíces con el *stemmer* de NLTK, de modo que «botella» y «botellas» coinciden con la misma raíz (`botell`). El **orden de la lista define la prioridad** cuando dos intenciones obtienen el mismo puntaje.

Las **reglas conversacionales** (ayuda, despedida, agradecimiento y saludo) se evalúan con expresiones regulares sólo cuando ninguna intención temática coincide, para que «Hola, ¿dónde tiro el vidrio?» responda sobre vidrio y no con un saludo.

In [3]:
# =====================================================================
# BASE DE CONOCIMIENTO
# Intenciones temáticas: identificador, descripción, ejemplos de consulta,
# palabras/frases clave y respuesta. El orden define la prioridad en empates.
# =====================================================================
INTENCIONES_TEMATICAS = [
    {
        'intencion': 'organicos',
        'descripcion': 'Separación de residuos orgánicos (restos de comida y jardín)',
        'ejemplos': ['¿Dónde tiro las cáscaras de fruta?', 'Tengo restos de comida y café molido'],
        'claves': ['organico', 'organicos', 'cascara', 'cascaras', 'restos de comida', 'comida', 'fruta', 'frutas',
                   'verdura', 'verduras', 'cafe', 'cascaron', 'huesos', 'tortilla', 'jardin', 'pasto', 'hojas secas', 'composta'],
        'respuesta': 'Los residuos orgánicos (cáscaras, restos de comida, café, cascarón y residuos de jardín) van en el '
                     'contenedor de orgánicos, que normalmente se identifica con el color verde. Escúrrelos bien y no los '
                     'mezcles con plástico, papel ni bolsas. Si tienes espacio, puedes aprovecharlos para hacer composta. '
                     'Confirma el color del contenedor y el día de recolección con tu municipio.',
    },
    {
        'intencion': 'plastico',
        'descripcion': 'Separación de plásticos (PET, envases, bolsas, unicel)',
        'ejemplos': ['¿Cómo separo las botellas de plástico?', '¿El unicel se recicla?'],
        'claves': ['plastico', 'plasticos', 'pet', 'botella', 'botellas', 'botella de plastico', 'envase', 'envases',
                   'bolsa', 'bolsas', 'unicel', 'popote', 'popotes', 'tapa', 'tapas', 'tupper', 'polietileno', 'pvc'],
        'respuesta': 'Los plásticos reciclables (botellas PET, envases de HDPE como los de champú o detergente y las tapas) '
                     'van en inorgánicos reciclables: enjuágalos, aplástalos y sepáralos de otros materiales. Las bolsas, '
                     'los popotes y el unicel tienen poca aceptación en los centros de acopio; pregunta si los reciben y, '
                     'si no, deposítalos en inorgánicos no reciclables.',
    },
    {
        'intencion': 'papel_carton',
        'descripcion': 'Separación de papel y cartón',
        'ejemplos': ['Tengo cajas de cartón y periódicos viejos', '¿Dónde van las revistas?'],
        'claves': ['papel', 'carton', 'periodico', 'periodicos', 'revista', 'revistas', 'caja', 'cajas', 'cuaderno',
                   'cuadernos', 'libro', 'libros', 'hojas de papel', 'folleto', 'folletos'],
        'respuesta': 'El papel, el cartón, los periódicos, las revistas y las cajas van en inorgánicos reciclables, siempre '
                     'limpios y secos. Dobla o aplana las cajas para ahorrar espacio. El papel manchado de grasa (como las '
                     'cajas de pizza sucias), el papel encerado y las servilletas usadas no se reciclan: van en inorgánicos '
                     'no reciclables.',
    },
    {
        'intencion': 'vidrio',
        'descripcion': 'Separación de vidrio (botellas, frascos) y vidrio no reciclable',
        'ejemplos': ['¿Los frascos de vidrio se reciclan?', '¿Qué hago con un espejo roto?'],
        'claves': ['vidrio', 'vidrios', 'frasco', 'frascos', 'cristal', 'botella de vidrio', 'envase de vidrio', 'foco',
                   'focos', 'espejo', 'espejos', 'ceramica', 'loza', 'ventana'],
        'respuesta': 'Las botellas y los frascos de vidrio van en inorgánicos reciclables: enjuágalos y retira las tapas. '
                     'No los mezcles con focos, espejos, cerámica, loza ni vidrio de ventanas, que tienen otra composición '
                     'y no se reciclan con el vidrio común; envuélvelos para evitar cortaduras y deposítalos en inorgánicos '
                     'no reciclables (los focos ahorradores van a un punto de manejo especial).',
    },
    {
        'intencion': 'metal',
        'descripcion': 'Separación de metales (latas, aluminio, chatarra)',
        'ejemplos': ['¿Dónde tiro las latas de aluminio?', 'Tengo latas de hojalata y chatarra'],
        'claves': ['lata', 'latas', 'aluminio', 'metal', 'metales', 'hojalata', 'chatarra', 'cobre', 'fierro', 'hierro',
                   'acero', 'aerosol', 'aerosoles'],
        'respuesta': 'Las latas de aluminio y hojalata, las tapas metálicas y la chatarra pequeña van en inorgánicos '
                     'reciclables. Enjuaga y aplasta las latas para reducir volumen. Los aerosoles solo se reciclan si están '
                     'completamente vacíos; si contienen restos de pintura o químicos, trátalos como residuo peligroso.',
    },
    {
        'intencion': 'electronicos',
        'descripcion': 'Residuos electrónicos y pilas (manejo especial)',
        'ejemplos': ['Tengo pilas usadas y un celular viejo', '¿Dónde tiro un cargador que ya no sirve?'],
        'claves': ['pila', 'pilas', 'bateria', 'baterias', 'celular', 'celulares', 'telefono', 'computadora', 'laptop',
                   'cargador', 'cargadores', 'electronico', 'electronicos', 'cable', 'cables', 'television', 'televisor',
                   'pantalla', 'impresora', 'audifonos', 'tablet'],
        'respuesta': 'Las pilas, baterías, celulares, cargadores, cables y demás aparatos electrónicos son residuos de '
                     'manejo especial: no van en la basura común porque contienen metales pesados. Guárdalos en una bolsa o '
                     'caja aparte y llévalos a un punto de acopio de electrónicos (por ejemplo, jornadas como el '
                     'Reciclatrón en la Ciudad de México) o a los contenedores de pilas que hay en escuelas, plazas y '
                     'tiendas participantes.',
    },
    {
        'intencion': 'peligrosos',
        'descripcion': 'Residuos peligrosos (medicamentos, aceites, pinturas, químicos)',
        'ejemplos': ['Se me caducaron unos medicamentos', '¿Qué hago con el aceite de cocina usado?'],
        'claves': ['medicina', 'medicinas', 'medicamento', 'medicamentos', 'pastillas', 'jarabe', 'aceite', 'pintura',
                   'pinturas', 'solvente', 'solventes', 'quimico', 'quimicos', 'jeringa', 'jeringas', 'aguja', 'agujas',
                   'insecticida', 'termometro', 'gasolina', 'thinner'],
        'respuesta': 'Los medicamentos caducos, aceites, pinturas, solventes, jeringas y productos químicos son residuos '
                     'peligrosos: nunca los tires al drenaje ni a la basura común. Los medicamentos se entregan en los '
                     'contenedores del programa SINGREM que hay en farmacias; el aceite de cocina usado, en frascos '
                     'cerrados, en centros de acopio; y las pinturas, solventes y jeringas, en las jornadas de residuos '
                     'peligrosos de tu municipio.',
    },
    {
        'intencion': 'sanitarios',
        'descripcion': 'Residuos sanitarios e inorgánicos no reciclables',
        'ejemplos': ['¿Dónde van los pañales?', '¿El papel higiénico se recicla?'],
        'claves': ['pañal', 'pañales', 'papel higienico', 'toalla sanitaria', 'toallas sanitarias', 'toallitas',
                   'cubrebocas', 'colilla', 'colillas', 'cigarro', 'cigarros', 'chicle', 'chicles', 'servilleta',
                   'servilletas', 'curita', 'curitas', 'hisopo', 'hisopos', 'rastrillo'],
        'respuesta': 'Los pañales, el papel higiénico, las toallas sanitarias, las toallitas húmedas, los cubrebocas, las '
                     'colillas y los chicles son residuos sanitarios, es decir, inorgánicos no reciclables. Colócalos en '
                     'una bolsa cerrada e identificada y deposítalos en el contenedor de no reciclables; nunca los mezcles '
                     'con los orgánicos ni con los reciclables.',
    },
    {
        'intencion': 'recoleccion',
        'descripcion': 'Días y horarios de recolección',
        'ejemplos': ['¿Qué día pasa el camión de la basura?', '¿Cuál es el horario de recolección?'],
        'claves': ['recoleccion', 'recolectan', 'recogen', 'camion', 'camion de la basura', 'horario', 'horarios',
                   'cuando pasa', 'que dia pasa', 'calendario'],
        'respuesta': 'Los días y horarios de recolección dependen de tu colonia y municipio, y ReciclaBot no tiene acceso a '
                     'esa información. Consulta el calendario de recolección en el sitio web o las redes sociales de tu '
                     'alcaldía o municipio, o pregunta al personal del camión. En muchas localidades los orgánicos y los '
                     'inorgánicos se recogen en días distintos.',
    },
    {
        'intencion': 'acopio',
        'descripcion': 'Dónde entregar materiales reciclables',
        'ejemplos': ['¿Hay un centro de acopio cerca?', '¿Dónde puedo llevar mis reciclables?'],
        'claves': ['acopio', 'centro de acopio', 'centros de acopio', 'llevar', 'donde puedo llevar', 'entregar',
                   'reciclatron', 'punto limpio', 'punto de reciclaje', 'contenedor', 'contenedores', 'maquina'],
        'respuesta': 'Para entregar tus reciclables busca los centros de acopio, puntos limpios o jornadas de reciclaje de '
                     'tu municipio; además, algunos supermercados y plazas tienen máquinas o contenedores para PET, latas y '
                     'pilas. Lleva los materiales limpios, secos y separados por tipo. Revisa el directorio de centros de '
                     'acopio de tu localidad para ubicar el más cercano.',
    },
    {
        'intencion': 'consejos',
        'descripcion': 'Consejos para reducir y reutilizar',
        'ejemplos': ['Dame consejos para generar menos basura', '¿Cómo puedo reducir mi basura?'],
        'claves': ['consejo', 'consejos', 'tip', 'tips', 'reducir', 'reutilizar', 'reusar', 'menos basura', 'evitar',
                   'ahorrar', 'tres erres', '3r', 'sustentable', 'sostenible', 'recomendaciones'],
        'respuesta': 'Tres acciones sencillas: 1) Reduce: lleva bolsa y termo reutilizables y evita productos con exceso '
                     'de empaque. 2) Reutiliza: los frascos, las cajas y las bolsas pueden tener una segunda vida antes de '
                     'desecharse. 3) Separa desde casa: orgánicos, reciclables y no reciclables en contenedores distintos. '
                     'Comprar a granel y hacer composta reducen todavía más tu basura.',
    },
    {
        'intencion': 'importancia',
        'descripcion': 'Qué es el reciclaje y por qué es importante separar',
        'ejemplos': ['¿Por qué es importante reciclar?', '¿Qué es el reciclaje?'],
        'claves': ['importancia', 'importante', 'por que reciclar', 'para que sirve', 'que es reciclar',
                   'que es el reciclaje', 'beneficio', 'beneficios', 'medio ambiente', 'ambiente', 'contaminacion',
                   'planeta', 'por que separar'],
        'respuesta': 'Separar y reciclar reduce la cantidad de residuos que llegan a los rellenos sanitarios, permite '
                     'recuperar materiales como PET, aluminio, papel y vidrio, y disminuye la contaminación y el consumo de '
                     'recursos naturales. La separación correcta desde casa es el primer paso para que el reciclaje sea '
                     'posible: un material contaminado con restos de comida muchas veces ya no se puede recuperar.',
    },
]

# Reglas conversacionales: se evalúan con expresiones regulares sobre el texto
# normalizado, sólo cuando ninguna intención temática obtiene puntaje.
REGLAS_CONVERSACIONALES = [
    {
        'intencion': 'ayuda',
        'descripcion': 'Menú de temas disponibles',
        'ejemplos': ['¿Qué puedes hacer?', 'menú'],
        'patron': r'\b(ayuda|ayudame|menu|opciones|que puedes hacer|que sabes hacer|temas|instrucciones|como funcionas|reciclar|reciclaje|reciclo)\b',
        'respuesta': 'Puedo orientarte sobre cómo separar: orgánicos, plásticos, papel y cartón, vidrio, metales, '
                     'electrónicos y pilas, residuos peligrosos y residuos sanitarios. También sobre dónde llevar los '
                     'reciclables, la recolección y consejos para generar menos basura. Escribe "salir" para terminar.',
    },
    {
        'intencion': 'despedida',
        'descripcion': 'Cierre de la conversación',
        'ejemplos': ['Adiós', 'salir'],
        'patron': r'\b(adios|salir|hasta luego|hasta pronto|bye|chao|nos vemos|me voy)\b',
        'respuesta': '¡Hasta luego! Gracias por usar ReciclaBot. Recuerda: separar bien es reciclar mejor.',
    },
    {
        'intencion': 'agradecimiento',
        'descripcion': 'Agradecimiento del usuario',
        'ejemplos': ['Gracias', 'Perfecto, muy amable'],
        'patron': r'\b(gracias|te agradezco|muy amable|perfecto|excelente|genial)\b',
        'respuesta': '¡Con gusto! Cada residuo bien separado cuenta. ¿Tienes otra duda sobre reciclaje?',
    },
    {
        'intencion': 'saludo',
        'descripcion': 'Inicio de la conversación',
        'ejemplos': ['Hola', 'Buenos días'],
        'patron': r'^(hola|buenos dias|buenas tardes|buenas noches|buen dia|hey|que tal|saludos|buenas)\b',
        'respuesta': '¡Hola! Soy ReciclaBot, tu asistente para separar correctamente los residuos en casa. Pregúntame, '
                     'por ejemplo, "¿dónde tiro las pilas?" o "¿cómo separo el plástico?". Escribe "ayuda" para ver '
                     'todos los temas.',
    },
]

# Intenciones especiales: entrada vacía y consulta no reconocida.
RESPUESTAS_ESPECIALES = {
    'vacio': 'No escribiste ninguna consulta. Cuéntame qué residuo quieres separar o escribe "ayuda" para ver los temas.',
    'desconocida': 'No logré identificar tu consulta. Puedo ayudarte con orgánicos, plásticos, papel y cartón, vidrio, '
                   'metales, electrónicos, residuos peligrosos, residuos sanitarios, centros de acopio, recolección y '
                   'consejos. Intenta mencionar el material, por ejemplo: "¿dónde tiro los frascos de vidrio?".',
}

print(f'Intenciones temáticas: {len(INTENCIONES_TEMATICAS)} | conversacionales: {len(REGLAS_CONVERSACIONALES)} '
      f'| especiales: {len(RESPUESTAS_ESPECIALES)} | total: '
      f'{len(INTENCIONES_TEMATICAS) + len(REGLAS_CONVERSACIONALES) + len(RESPUESTAS_ESPECIALES)}')

Intenciones temáticas: 12 | conversacionales: 4 | especiales: 2 | total: 18


### Motor de reglas
El motor procesa cada entrada en cuatro pasos: normalización, preprocesamiento con NLTK (tokenización, eliminación de *stopwords* y *stemming*), identificación de la intención y selección de la respuesta.

In [4]:
import re
import unicodedata
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import RegexpTokenizer

stemmer = SnowballStemmer('spanish')      # reduce las palabras a su raíz: botellas -> botell
tokenizador = RegexpTokenizer(r'\w+')     # separa palabras y descarta signos de puntuación


def normalizar(texto):
    """Convierte a minúsculas, elimina acentos y espacios repetidos."""
    texto = unicodedata.normalize('NFD', texto.lower().strip())
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    return re.sub(r'\s+', ' ', texto)


try:
    STOPWORDS = {normalizar(p) for p in stopwords.words('spanish')}
except LookupError:   # respaldo si no se pudo descargar el corpus
    STOPWORDS = {'de', 'la', 'el', 'los', 'las', 'un', 'una', 'y', 'o', 'que', 'en', 'a', 'con', 'para', 'por',
                 'mi', 'mis', 'me', 'se', 'es', 'al', 'del', 'lo', 'le', 'su', 'sus', 'ya', 'como', 'donde', 'cuando'}


def preprocesar(texto):
    """Normaliza, tokeniza, elimina stopwords y aplica stemming (NLTK)."""
    tokens = tokenizador.tokenize(normalizar(texto))
    return [stemmer.stem(t) for t in tokens if t not in STOPWORDS]


# Precálculo de las claves: palabras sueltas -> raíz; frases -> texto normalizado.
for _intencion in INTENCIONES_TEMATICAS:
    _intencion['raices'] = {stemmer.stem(normalizar(c)) for c in _intencion['claves'] if ' ' not in c}
    _intencion['frases'] = [normalizar(c) for c in _intencion['claves'] if ' ' in c]


def puntuar(texto_normalizado, raices_usuario, intencion):
    """Puntaje = raíces coincidentes (1 punto) + frases coincidentes (2 puntos)."""
    puntos = sum(1 for r in raices_usuario if r in intencion['raices'])
    puntos += sum(2 for f in intencion['frases'] if f in texto_normalizado)
    return puntos


def identificar_intencion(texto):
    """Entrada del usuario -> (intención, respuesta, detalle del análisis)."""
    limpio = normalizar(texto)
    if not limpio:
        return 'vacio', RESPUESTAS_ESPECIALES['vacio'], {'raices': [], 'puntaje': 0}
    raices = preprocesar(texto)
    # 1) Intenciones temáticas: gana el mayor puntaje; en empate, la primera de la lista.
    puntajes = {i['intencion']: puntuar(limpio, raices, i) for i in INTENCIONES_TEMATICAS}
    mejor = max(INTENCIONES_TEMATICAS, key=lambda i: puntajes[i['intencion']])
    if puntajes[mejor['intencion']] > 0:
        return mejor['intencion'], mejor['respuesta'], {'raices': raices, 'puntaje': puntajes[mejor['intencion']]}
    # 2) Reglas conversacionales por expresión regular.
    for regla in REGLAS_CONVERSACIONALES:
        if re.search(regla['patron'], limpio):
            return regla['intencion'], regla['respuesta'], {'raices': raices, 'puntaje': 0}
    # 3) Consulta no reconocida.
    return 'desconocida', RESPUESTAS_ESPECIALES['desconocida'], {'raices': raices, 'puntaje': 0}


def responder(texto):
    """Devuelve un diccionario con la entrada, la intención identificada y la respuesta."""
    intencion, respuesta, detalle = identificar_intencion(texto)
    return {'entrada': texto, 'raices': detalle['raices'], 'puntaje': detalle['puntaje'],
            'intencion': intencion, 'respuesta': respuesta}


def ejecutar_pruebas(conversaciones, mostrar=True):
    """Ejecuta conversaciones [(entrada, intención esperada), ...] y devuelve los registros."""
    registros = []
    for numero, conversacion in enumerate(conversaciones, 1):
        if mostrar:
            print(f'\n=== Conversación {numero:02d} ===')
        for entrada, esperada in conversacion:
            salida = responder(entrada)
            correcto = salida['intencion'] == esperada
            registros.append({'conversacion': numero, 'entrada': entrada, 'raices': ' '.join(salida['raices']),
                              'esperada': esperada, 'intencion': salida['intencion'], 'correcto': correcto,
                              'respuesta': salida['respuesta']})
            if mostrar:
                print(f'Usuario   : {entrada!r}')
                print(f'ReciclaBot: {salida["respuesta"]}')
                print(f'Intención : {salida["intencion"]} | esperada: {esperada} | '
                      f'{"OK" if correcto else "REVISAR"}')
    return registros


# Resumen del diseño conversacional
print(f'{"Intención":<16}{"Ejemplo de entrada":<44}Respuesta (inicio)')
for i in INTENCIONES_TEMATICAS + REGLAS_CONVERSACIONALES:
    print(f'{i["intencion"]:<16}{i["ejemplos"][0]:<44}{i["respuesta"][:55]}...')
for nombre, respuesta in RESPUESTAS_ESPECIALES.items():
    print(f'{nombre:<16}{"(sin coincidencia)":<44}{respuesta[:55]}...')
print('\nEjemplo de preprocesamiento:', '¿Dónde tiro las cáscaras de fruta?', '->',
      preprocesar('¿Dónde tiro las cáscaras de fruta?'))

Intención       Ejemplo de entrada                          Respuesta (inicio)
organicos       ¿Dónde tiro las cáscaras de fruta?          Los residuos orgánicos (cáscaras, restos de comida, caf...
plastico        ¿Cómo separo las botellas de plástico?      Los plásticos reciclables (botellas PET, envases de HDP...
papel_carton    Tengo cajas de cartón y periódicos viejos   El papel, el cartón, los periódicos, las revistas y las...
vidrio          ¿Los frascos de vidrio se reciclan?         Las botellas y los frascos de vidrio van en inorgánicos...
metal           ¿Dónde tiro las latas de aluminio?          Las latas de aluminio y hojalata, las tapas metálicas y...
electronicos    Tengo pilas usadas y un celular viejo       Las pilas, baterías, celulares, cargadores, cables y de...
peligrosos      Se me caducaron unos medicamentos           Los medicamentos caducos, aceites, pinturas, solventes,...
sanitarios      ¿Dónde van los pañales?                     Los pañales, el papel higién

### Validación de los ejemplos de diseño
Cada ejemplo definido en la base de conocimiento debe ser reconocido con su propia intención. Esta celda comprueba la coherencia entre el diseño conversacional y la implementación.

In [5]:
fallos = []
for intencion in INTENCIONES_TEMATICAS + REGLAS_CONVERSACIONALES:
    for ejemplo in intencion['ejemplos']:
        obtenida = identificar_intencion(ejemplo)[0]
        estado = 'OK' if obtenida == intencion['intencion'] else 'REVISAR'
        print(f'{estado:<8}{ejemplo!r:<48} -> {obtenida}')
        if estado != 'OK':
            fallos.append((ejemplo, intencion['intencion'], obtenida))
print(f'\nEjemplos validados: {sum(len(i["ejemplos"]) for i in INTENCIONES_TEMATICAS + REGLAS_CONVERSACIONALES)} '
      f'| fallos: {len(fallos)}')
assert not fallos, f'Ejemplos que requieren revisión: {fallos}'

OK      '¿Dónde tiro las cáscaras de fruta?'             -> organicos
OK      'Tengo restos de comida y café molido'           -> organicos
OK      '¿Cómo separo las botellas de plástico?'         -> plastico
OK      '¿El unicel se recicla?'                         -> plastico
OK      'Tengo cajas de cartón y periódicos viejos'      -> papel_carton
OK      '¿Dónde van las revistas?'                       -> papel_carton
OK      '¿Los frascos de vidrio se reciclan?'            -> vidrio
OK      '¿Qué hago con un espejo roto?'                  -> vidrio
OK      '¿Dónde tiro las latas de aluminio?'             -> metal
OK      'Tengo latas de hojalata y chatarra'             -> metal
OK      'Tengo pilas usadas y un celular viejo'          -> electronicos
OK      '¿Dónde tiro un cargador que ya no sirve?'       -> electronicos
OK      'Se me caducaron unos medicamentos'              -> peligrosos
OK      '¿Qué hago con el aceite de cocina usado?'       -> peligrosos
OK      '¿Dónde van lo

## 4. Doce conversaciones distintas de prueba
Las conversaciones cubren las 18 intenciones e incluyen mayúsculas, acentos, texto vacío, una consulta fuera de alcance, una consulta con error ortográfico, una pregunta que depende del contexto anterior y una consulta con dos temas. Las intenciones esperadas se declaran **antes** de ejecutar las pruebas. La tasa obtenida describe únicamente estos casos diseñados; no es una medida de precisión con usuarios reales.

In [10]:
# Cada conversación es una lista de turnos (entrada del usuario, intención esperada).
CONVERSACIONES = [
    [('Hola', 'saludo'), ('¿Dónde tiro las cáscaras de fruta?', 'organicos'), ('Gracias', 'agradecimiento')],
    [('Buenos días', 'saludo'), ('¿Cómo separo las botellas de plástico?', 'plastico'),
     ('¿Y las bolsas de plástico?', 'plastico'), ('Adiós', 'despedida')],
    [('Tengo cajas de cartón y periódicos viejos', 'papel_carton'),
     ('¿Qué hago con una caja de pizza sucia?', 'papel_carton')],
    [('¿Los frascos de vidrio se reciclan?', 'vidrio'), ('¿Y los focos?', 'vidrio'), ('salir', 'despedida')],
    [('¿DÓNDE TIRO LAS LATAS DE ALUMINIO?', 'metal'), ('gracias', 'agradecimiento')],
    [('Tengo pilas usadas y un celular viejo', 'electronicos'), ('¿Dónde las puedo llevar?', 'acopio')],
    [('Se me caducaron unos medicamentos', 'peligrosos'), ('¿Y el aceite de cocina usado?', 'peligrosos')],
    [('¿Dónde van los pañales y el papel higiénico?', 'sanitarios'), ('Hasta luego', 'despedida')],
    [('¿Qué día pasa el camión de la basura?', 'recoleccion'), ('¿Hay un centro de acopio cerca?', 'acopio')],
    [('¿Qué puedes hacer?', 'ayuda'), ('Dame consejos para generar menos basura', 'consejos'),
     ('Perfecto, gracias', 'agradecimiento')],
    [('   ', 'vacio'), ('¿Por qué es importante reciclar?', 'importancia')],
    [('¿Qué clima habrá mañana?', 'desconocida'), ('Quiero tirar plastco', 'desconocida'),
     ('¿Dónde tiro botellas de plástico y latas?', 'plastico')],
]
print(f'Conversaciones: {len(CONVERSACIONES)} | turnos: {sum(len(c) for c in CONVERSACIONES)}')

Conversaciones: 12 | turnos: 30


In [7]:
registros = ejecutar_pruebas(CONVERSACIONES)
aciertos = sum(r['correcto'] for r in registros)
print(f'\nResultado: {aciertos}/{len(registros)} turnos coinciden con la intención esperada.')
print('Intenciones cubiertas:', len({r['intencion'] for r in registros}))

import pandas as pd
tabla = pd.DataFrame(registros)[['conversacion', 'entrada', 'raices', 'esperada', 'intencion', 'correcto']]
try:
    display(tabla)
except NameError:
    print(tabla.to_string(index=False))
assert aciertos == len(registros), 'Hay turnos que requieren revisión.'


=== Conversación 01 ===
Usuario   : 'Hola'
ReciclaBot: ¡Hola! Soy ReciclaBot, tu asistente para separar correctamente los residuos en casa. Pregúntame, por ejemplo, "¿dónde tiro las pilas?" o "¿cómo separo el plástico?". Escribe "ayuda" para ver todos los temas.
Intención : saludo | esperada: saludo | OK
Usuario   : '¿Dónde tiro las cáscaras de fruta?'
ReciclaBot: Los residuos orgánicos (cáscaras, restos de comida, café, cascarón y residuos de jardín) van en el contenedor de orgánicos, que normalmente se identifica con el color verde. Escúrrelos bien y no los mezcles con plástico, papel ni bolsas. Si tienes espacio, puedes aprovecharlos para hacer composta. Confirma el color del contenedor y el día de recolección con tu municipio.
Intención : organicos | esperada: organicos | OK
Usuario   : 'Gracias'
ReciclaBot: ¡Con gusto! Cada residuo bien separado cuenta. ¿Tienes otra duda sobre reciclaje?
Intención : agradecimiento | esperada: agradecimiento | OK

=== Conversación 02 ===
Usuario  

,conversacion,entrada,raices,esperada,intencion,correcto
0,1,Hola,hol,saludo,saludo,True
1,1,¿Dónde tiro las cáscaras de fruta?,tir casc frut,organicos,organicos,True
2,1,Gracias,graci,agradecimiento,agradecimiento,True
3,2,Buenos días,buen dias,saludo,saludo,True
4,2,¿Cómo separo las botellas de plástico?,separ botell plastic,plastico,plastico,True
5,2,¿Y las bolsas de plástico?,bols plastic,plastico,plastico,True
6,2,Adiós,adi,despedida,despedida,True
7,3,Tengo cajas de cartón y periódicos viejos,caj carton period viej,papel_carton,papel_carton,True
8,3,¿Qué hago con una caja de pizza sucia?,hag caj pizz suci,papel_carton,papel_carton,True
9,4,¿Los frascos de vidrio se reciclan?,frasc vidri recicl,vidrio,vidrio,True


## 5. Conversación manual (opcional)
Cambia `ACTIVAR_CHAT` a `True` y ejecuta la celda. Escribe `salir` para terminar. Se deja desactivada para que **Ejecutar todas** no se quede esperando una entrada.

In [8]:
ACTIVAR_CHAT = False
if ACTIVAR_CHAT:
    print('ReciclaBot: ¡Hola! Escribe tu consulta sobre residuos o "salir" para terminar.')
    while True:
        entrada = input('Tú: ')
        salida = responder(entrada)
        print(f'ReciclaBot [{salida["intencion"]}]: {salida["respuesta"]}')
        if salida['intencion'] == 'despedida':
            break
else:
    print('Cambia ACTIVAR_CHAT a True para conversar manualmente.')

Cambia ACTIVAR_CHAT a True para conversar manualmente.


## 6. Exportación de evidencias
Genera un archivo CSV y uno de texto con las conversaciones de prueba y los empaqueta en un ZIP. Cambia `DESCARGAR` a `True` para descargarlo desde Colab. Complementa estos archivos con capturas de pantalla de las salidas del notebook.

In [9]:
import csv
import zipfile
from pathlib import Path
from datetime import datetime, timezone

carpeta = Path('evidencias_reciclabot')
carpeta.mkdir(exist_ok=True)
with (carpeta / 'pruebas.csv').open('w', newline='', encoding='utf-8-sig') as f:
    escritor = csv.DictWriter(f, fieldnames=list(registros[0]))
    escritor.writeheader()
    escritor.writerows(registros)
lineas = ['ReciclaBot - evidencias de ejecución', 'Ejecución UTC: ' + datetime.now(timezone.utc).isoformat(),
          f'Python: {sys.version.split()[0]} | NLTK: {nltk.__version__}', '']
for r in registros:
    lineas += [f"Conversación {r['conversacion']:02d} | Usuario: {r['entrada']!r}", 'ReciclaBot: ' + r['respuesta'],
               f"Intención: {r['intencion']} | Esperada: {r['esperada']} | Correcto: {r['correcto']}", '']
lineas.append(f'Resultado: {aciertos}/{len(registros)} turnos correctos.')
(carpeta / 'conversaciones.txt').write_text('\n'.join(lineas), encoding='utf-8')
with zipfile.ZipFile('Evidencias_ReciclaBot.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for archivo in carpeta.iterdir():
        z.write(archivo, arcname=archivo.name)
print('Evidencias generadas: Evidencias_ReciclaBot.zip ->', [a.name for a in carpeta.iterdir()])

DESCARGAR = False   # Cambia a True para descargar el ZIP desde Colab.
if DESCARGAR:
    from google.colab import files
    files.download('Evidencias_ReciclaBot.zip')

Evidencias generadas: Evidencias_ReciclaBot.zip -> ['conversaciones.txt', 'pruebas.csv']


## 7. Resultados, limitaciones y mejoras
**Funcionamiento observado.** Las doce conversaciones (30 turnos) se identifican con la intención esperada. El preprocesamiento con NLTK permite reconocer variantes morfológicas (singular/plural, mayúsculas, acentos) y la puntuación por claves resuelve consultas con más de una palabra relevante. La consulta fuera de alcance y la consulta con error ortográfico caen correctamente en la intención `desconocida`, que orienta al usuario sobre los temas disponibles.

**Limitaciones identificadas en las pruebas.**
1. **Sin contexto entre turnos:** tras «Tengo pilas usadas», la pregunta «¿Dónde las puedo llevar?» recibe la respuesta general de centros de acopio y no la específica de electrónicos.
2. **Sin tolerancia a errores ortográficos:** «plastco» no coincide con ninguna raíz y se responde como desconocida.
3. **Consultas con dos temas:** «botellas de plástico y latas» responde sólo sobre plástico (mayor puntaje); el segundo tema se ignora.
4. **Negaciones:** la palabra «no» se elimina como *stopword*, por lo que «no es orgánico» se interpreta como «orgánico».
5. **Información general:** las respuestas no consideran las reglas específicas de cada municipio.

**Mejoras relacionadas para las siguientes etapas.**
1. Conservar la última intención temática como contexto para resolver preguntas de seguimiento.
2. Agregar corrección ortográfica (distancia de edición con `nltk.edit_distance`) o un clasificador de intenciones entrenado con ejemplos.
3. Responder a todas las intenciones con puntaje mayor que cero o pedir una aclaración cuando haya empate.
4. Detectar negaciones antes de eliminar las *stopwords*.
5. Incorporar una base de datos por municipio e integrar la clasificación de imágenes de residuos y la interfaz web.


## Referencias
- Bird, S., Klein, E. y Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly Media. https://www.nltk.org/book/
- NLTK Project. (2024). *NLTK 3.9 documentation*. https://www.nltk.org/
- Python Software Foundation. (2024). *re — Regular expression operations*. https://docs.python.org/3/library/re.html
- Google. (2024). *Google Colaboratory*. https://colab.research.google.com/
- Secretaría de Medio Ambiente y Recursos Naturales. (2020). *Diagnóstico Básico para la Gestión Integral de los Residuos*. SEMARNAT. https://www.gob.mx/semarnat